In [0]:
# %sql
# use catalog lending_risk;
# drop database if exists bronze cascade;
# drop database if exists silver cascade;
# drop database if exists gold cascade; 

In [0]:
# --- 1. Global Config (Match your paths.py) ---
CATALOG = "lending_risk"
SCHEMAS = ["landing", "bronze", "silver", "gold"]

# --- 2. Namespace Initialization ---
print(f"Creating -- CATALOG => {CATALOG}")
spark.sql(f"CREATE CATALOG IF NOT EXISTS {CATALOG}")
spark.sql(f"USE CATALOG {CATALOG}")

for schema in SCHEMAS:
    print(f"Creating -- SCHEMA => {schema}")
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {schema}")

# --- 3. Volume Initialization (The Landing & Checkpoint Zones) ---
print(f"Creating -- VOLUME => {CATALOG}.landing.vol_landing")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.landing.vol_landing")

print(f"Creating -- VOLUME => {CATALOG}.bronze.vol_bronze")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.bronze.vol_bronze")
 
print(f"Creating -- VOLUME => {CATALOG}.silver.vol_silver")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.silver.vol_silver")

print(f"Creating -- VOLUME => {CATALOG}.gold.vol_gold")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.gold.vol_gold")

# --- 4. Directory Structure Creation ---
folders = ["customers", "loans", "defaulters", "repayments", "reference"]
LANDING_PATH = f"/Volumes/{CATALOG}/landing/vol_landing"

for folder in folders:
    folder_path = f"{LANDING_PATH}/{folder}"
    print(f"Ensuring Landing Directory => {folder_path}")
    dbutils.fs.mkdirs(folder_path)

# --- 5. Table Initialization ---
# We use dbutils.notebook.run to execute the layer-specific DDL scripts
print("Building Bronze Layer Tables...")
dbutils.notebook.run("01_create_tables_bronze", 600)

print("Building Silver Layer Tables...")
dbutils.notebook.run("02_create_tables_silver", 600)

print("Building Gold Layer Tables...")
dbutils.notebook.run("03_create_tables_gold", 600)

print("--- LENDING RISK ENVIRONMENT READY ---")